# 03. Adam Optimizer Demystified: From SGD to Adaptive Moments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/The-Pocket/PocketFlow-Tutorial-Video-Generator/blob/main/docs/llm/03_Adam_Optimizer_Demystified.ipynb)

> **Tutorial Overview**: Understand why standard SGD struggles in ravines and saddle points, how Momentum solves oscillations, how RMSprop scales learning rates, and how Adam combines both with Bias Correction.
> **Original Source**: `adam.md`

---


In [ ]:
# Setup & Imports
!pip install -q torch matplotlib numpy

import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)


# **Title: Give me 30 min, I will make the Adam Optimizer Click. Forever.**

## Intro

You know Gradient Descent. You know the formula: `new_weight = old_weight - learning_rate * gradient`. It's simple. It works.

But it's not what powers modern AI.

In every state-of-the-art model, in every high-performance training script, you see the same name: **Adam**. You're told to just use it. It's the default. It's "better."

But why?

You look up the algorithm and are hit with a wall of math. A black box of Greek letters and strange terms that feel impossibly complex.

*   Exponentially Weighted Moving Average (EWMA)
*   First and Second Moments
*   Bias Correction

It seems like a magic spell you're supposed to cast without understanding.

Here's the secret: **Adam isn't one complex idea. It's three simple ideas, bolted together to solve three specific problems.**

Give me 30 minutes. We will tear Adam down to its fundamental parts and rebuild it from the ground up. No skipped steps. No magic. By the end, you will have a deep, intuitive, and permanent understanding of every single component.

This is the algorithm you are about to master:

1.  **First Moment (Momentum):** $m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t$
2.  **Second Moment (Adaptive LR):** $v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2$
3.  **Bias Correction:** $\hat{m}_t = \frac{m_t}{1 - \beta_1^t}$ and $\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$
4.  **The Update:** $\theta_t = \theta_{t-1} - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$

That wall of math will look simple. You will see *why* each piece exists and the exact problem it solves.

Ready to make it click? Let's begin.

## **Part 1: The First Problem - Inefficient Progress**

**The Core Problem:** Gradient Descent is memoryless. As it gets closer to the minimum and the gradient shrinks, its steps become smaller and smaller, causing it to slow down dramatically.

Let's use the simple function `f(p) = p**2`. The minimum is at `p=0`, and the gradient is `f'(p) = 2p`. We start at `p=10` and use a learning rate of `η = 0.1`.

#### **Algorithm 1: Standard Gradient Descent**

The update rule is simple and direct.


```python
FOR each iteration:
  gradient = 2 * params
  params = params - 0.1 * gradient
```


This is our baseline—the slow, steady crawl.

| Iteration | Current `p` | Gradient `g = 2p` | Update `0.1 * g` | New `p` |
| :-------- | :---------- | :---------------- | :--------------- | :------ |
| 0         | 10.000      | 20.000            | 2.000            | 8.000   |
| 1         | 8.000       | 16.000            | 1.600            | 6.400   |
| 2         | 6.400       | 12.800            | 1.280            | 5.120   |
| 3         | 5.120       | 10.240            | 1.024            | 4.096   |
| 4         | 4.096       | 8.192             | 0.819            | 3.277   |

**Analysis of the Slowness:** The "Update" size is constantly shrinking: `2.0` → `1.6` → `1.28`... This is **deceleration**. The algorithm becomes less effective with every step.

#### **Algorithm 2: Gradient Descent with Momentum**

Now, let's add a `velocity` term with a more moderate `beta` of `0.5`. This will allow inertia to build up without running out of control.


```python
velocity = 0
FOR each iteration:
  gradient = 2 * params
  velocity = 0.5 * velocity + gradient
  params = params - 0.1 * velocity
```


Watch the difference in convergence.

| Iteration | Current `p` | Gradient `g` | Velocity `v = 0.5*v + g` | New `p` |
| :-------- | :---------- | :----------- | :--------------------------- | :------ |
| 0         | 10.000      | 20.000       | `0.5*0 + 20.0 = 20.000`      | 8.000   |
| 1         | 8.000       | 16.000       | `0.5*20.0 + 16.0 = 26.000`   | 5.400   |
| 2         | 5.400       | 10.800       | `0.5*26.0 + 10.8 = 23.800`   | 3.020   |
| 3         | 3.020       | 6.040        | `0.5*23.8 + 6.04 = 17.940`   | 1.226   |
| 4         | 1.226       | 2.452        | `0.5*17.94 + 2.45 = 11.422`  | 0.084   |

**Analysis of the Success:**
*   **Compare `p` at Iteration 4:** Standard Gradient Descent is still far away at `3.277`. Momentum is already at `0.084`, practically at the minimum. This is a clear, unambiguous win.
*   **Look at the `velocity`:** In step 1, the gradient was `16`, but the velocity was `26`. In step 2, the gradient was `10.8`, but the velocity was `23.8`. Because the gradients were all in the same direction, they accumulated, creating a much larger and more effective update step. This is **controlled acceleration**.
*   **No Instability:** Unlike the previous bad example, this version converges beautifully without any wild overshooting.

---
#### **Revisiting the Ravine: The Two Jobs of Momentum**

Now we can confidently state that Momentum is a superior algorithm. In a complex landscape like our 2D ravine (`f(p) = p[0]**2 + 50 * p[1]**2`), it performs two critical jobs simultaneously:

1.  **Accelerates:** In the shallow `p[0]` direction, the gradients are small but consistent. Momentum builds up velocity here—just like in our successful 1D example—speeding up progress along the valley floor.
2.  **Damps:** In the steep `p[1]` direction, the gradients are huge but constantly flip signs (`+150`, `-120`, etc.). When Momentum averages these opposing forces, they cancel each other out, which powerfully suppresses the wasteful zig-zagging.

Momentum intelligently uses its memory of past gradients to navigate more efficiently.

**Problem Solved:** We have a mechanism to fix Gradient Descent's inefficient, memoryless updates.

**But a new problem emerges:** While smarter, this approach still applies the same learning rate to every parameter. Isn't there a way to give each parameter its *own* adaptive learning rate from the start?

## **Part 2: The Second Problem - Inflexible Learning Rates**

**The Core Problem:** Momentum helps find a better direction, but it's still handicapped by a single, global learning rate. This fails when parameters have vastly different sensitivities.

Let's design a function where this failure is guaranteed:
`f(p) = 50 * p[0]**2 + p[1]**2`

The minimum is at `(0, 0)`. The gradient vector is:
*   `∂f/∂p[0] = 100 * p[0]`
*   `∂f/∂p[1] = 2 * p[1]`

The gradient for `p[0]` is **50 times stronger** than for `p[1]`. This means `p[0]` is an extremely "sensitive" parameter, while `p[1]` is "stubborn."

#### **Algorithm 1: Naive Gradient Descent**

To prevent the update for the sensitive `p[0]` from exploding, we are forced to choose a tiny learning rate. Let's use `η = 0.01`. We will start at `p = (1.5, 10.0)`.
